# bridge_player_position.py

## Purpose
Bridge table that links players to all their positions (primary + alternate).

## Sources
- Silver: `fifa.silver.dim_player`, `fifa.silver.dim_position`

## Business Rules
- Each player has exactly 1 primary position
- Each player may have 0..N alternate positions (space-separated string)
- Output: one row per (`player_snapshot_key`, `position_id`, `position_type`)

## Strategy
- Overwrite mode (small table, rebuilt on every run)
- Primary positions: direct FK from `dim_player`
- Alternate positions: exploded from the string, mapped via `dim_position`

## Output
- Table: `fifa.silver.bridge_player_position`

## Columns
- `player_snapshot_key` (STRING)
- `player_id` (INT)
- `position_id` (INT)
- `position_type` (STRING): `'primary'` or `'alternate'`

## Why a bridge table?

A player can have multiple positions (primary + alternates), and a position can be played by many players. That's a **many-to-many** relationship.

In a star schema, M:N relationships are resolved with a **bridge table**:

dim_position 

▼

bridge_player_position 

▼

dim_player


## Primary vs alternate

| Type | Source | Description |
|---|---|---|
| `primary` | `dim_player[position_id]` | The player's main position |
| `alternate` | `dim_player[alternate_positions]` | Space-separated list of extra positions |

Both are combined with `unionByName` into a single bridge table.

## Why `player_snapshot_key`?

The bridge table links to `dim_player` via `player_snapshot_key` (not just `player_id`) because:

1. `player_id` alone is not unique in `dim_player` (a player has one row per edition).
2. The relationship in Power BI needs a unique key.

## Important: always keep `player_snapshot_key`

In every `.select()` after the `explode`, `player_snapshot_key` must be preserved. Otherwise, the link to `dim_player` is lost.


In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, regexp_extract, when, concat,trim, explode, split, pmod, xxhash64

# --- Load dimensions ---
dim_player = spark.read.table("fifa.silver.dim_player").filter(col("Is_current") == True)
dim_position = spark.read.table("fifa.silver.dim_position")

# --- Primary position ---
primary = (
    dim_player
    .select("player_snapshot_key", "player_id", "position_id")
    .filter(col("position_id").isNotNull())
    .withColumn("position_type", lit("primary"))
)

# --- Alternate positions (exploded) ---
alternates = (
    dim_player
    .filter(col("alternate_positions").isNotNull() & (col("alternate_positions") != ""))
    .select(
        "player_snapshot_key",
        "player_id",
        explode(split(col("alternate_positions"), " ")).alias("position_code_raw")
    )
    .select(
        col("player_snapshot_key"), 
        col("player_id"),
        trim(col("position_code_raw")).alias("position_code")
    )
    .filter(col("position_code") != "")
    .join(dim_position.select("position_id", "position_code"), on="position_code", how="left")
    .select("player_snapshot_key","player_id", "position_id")
    .withColumn("position_type", lit("alternate"))
)

# --- Union of primary and alternate ---
bridge = primary.unionByName(alternates).distinct()


# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.bridge_player_position (
    player_snapshot_key STRING,
    player_id INT,
    position_id INT,
    position_type STRING
)
USING DELTA;
""")

# Overwrite the table
(bridge
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable("fifa.silver.bridge_player_position")
)

print(f"bridge_player_position creada con {spark.read.table('fifa.silver.bridge_player_position').count()} filas")

LIMIT 10 of the bridge_player_position table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
select 
  *
from fifa.silver.bridge_player_position order by player_id limit 10;